# Data preprocessing and corpus aggregation
- Data preprocessing
    - packages and dependencies
    - import raw data
    - filtering (CVs; file formats `.doc`, `.pdf`, and `.docx`)
    - text extraction, normalization, standardization
    - text from corpus table to actual `.txt`
- Data aggregation
    - exploratory aggregation (for EDA)
    - corpus table
    - corpus table quality checks


## Packages


In [ ]:
!pip install -q pymupdf python-docx pandas

!apt-get update -qq
!apt-get install -y -qq libreoffice

!soffice --version

In [ ]:
import pandas as pd
import numpy as np
import fitz ## PyMuPDF
import re
import hashlib
import json
import time
import subprocess
import matplotlib.pyplot as plt

from pathlib import Path
from docx import Document
from google.colab import drive
from IPython.display import display, HTML

## Import and locate data


In [ ]:
## import drive
drive.mount('/content/drive')

## define root path
BASE = Path("/content/drive/MyDrive/data_youthnex")

In [ ]:
## define cv folder names
CV_FOLDER_NAMES = {"cvs", "cv received"}

## search yearly directories for cv folders
for y in range(2020, 2026):
    year_path = BASE / str(y)

    matches = [
        p for p in year_path.rglob("*")
        if p.is_dir() and p.name.lower() in CV_FOLDER_NAMES
    ]

    ## find cv folders (nested ok)
    if matches:
        print(y, "found:")
        for m in matches:
            print("  ", m)
    else:
        print(y, "not found")

## Exploratory aggregation (useful for EDA)

In [ ]:

## cv folder and .ext aggregation
rows = []
folder_hits = []

## process loop
for y in range(2020, 2026):
    year = str(y)
    year_path = BASE / year

    ## missing year directory check
    if not year_path.exists():
        rows.append({"year": year, "ext": "(year_missing)"})
        folder_hits.append({"year": year, "cv_folders_found": 0})
        continue

    ## find cv folders under that year (nested ok)
    matches = [
        p for p in year_path.rglob("*")
        if p.is_dir() and p.name.strip().lower() in CV_FOLDER_NAMES
    ]
    folder_hits.append({"year": year, "cv_folders_found": len(matches)})

    ## missing cv folder check
    if not matches:
        rows.append({"year": year, "ext": "(no_cv_folder_found)"})
        continue

    ## count file extensions under found folders
    for folder in matches:
        for f in folder.rglob("*"):
            if f.is_file():
                ext = f.suffix.lower() if f.suffix else "(no_ext)"
                rows.append({"year": year, "ext": ext})

## create rows dataframe
df = pd.DataFrame(rows)
pivot = (
    df.groupby(["year", "ext"]).size()
      .unstack(fill_value=0)
      .reset_index()
      .sort_values("year")
)

## add extension totals
ext_cols = [c for c in pivot.columns if c != "year"]
pivot["TOTAL_FILES"] = pivot[ext_cols].sum(axis=1)

## add number of CV folders found from creating folder_hits df
hits_df = pd.DataFrame(folder_hits)
pivot = pivot.merge(hits_df, on="year", how="left")

pivot

### Visualizations for exploratory aggregation

In [ ]:
## exploratory aggregation visuals -- file types and counts
## wanted extension assignment
wanted_exts = [".doc", ".docx", ".pdf"]

## ensure columns exist even year has 0 of that type
for ext in wanted_exts:
    if ext not in pivot.columns:
        pivot[ext] = 0

## sort by year
plot_df = pivot.copy()
plot_df = plot_df[pd.to_numeric(plot_df["year"], errors="coerce").notna()]


## chart 1: total CVs per year .doc .docx or .pdf only

plot_df["DOC_DOCX_PDF_TOTAL"] = plot_df[wanted_exts].sum(axis=1)

plt.figure(figsize=(10, 4.5))
plt.bar(plot_df["year"].astype(str), plot_df["DOC_DOCX_PDF_TOTAL"])
plt.title("Total CVs per Year")
plt.xlabel("Year")
plt.ylabel("Number of CVs")
plt.tight_layout()
plt.show()


## chart 2: stacked bars
## colored based on extension type .doc .docx or .pdf only

plt.figure(figsize=(10, 4.8))
bottom = None

for ext in wanted_exts:
    vals = plot_df[ext].to_numpy()
    if bottom is None:
        plt.bar(plot_df["year"].astype(str), vals, label=ext)
        bottom = vals.copy()
    else:
        plt.bar(plot_df["year"].astype(str), vals, bottom=bottom, label=ext)
        bottom = bottom + vals

plt.title("CV File Count per Year by Extension (.doc/.docx/.pdf)")
plt.xlabel("Year")
plt.ylabel("File Count")
plt.legend(title="Extension", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Filtering, extraction, normalization, standardization

### Setup/configuration


In [ ]:
## path setup
BASE_ORIG = BASE ## alias for clarity: original data root
BASE_CONVERTED = Path("/content/drive/MyDrive/data_youthnex_docx_converted")

## constants
CV_FOLDER_NAMES = {"cvs", "cv received"}
ALLOWED_EXTS = {".pdf", ".docx"} ## extraction
CONVERT_EXTS = {".doc"} ## conversion

## initialize rows, seen_doc_ids
rows = []
seen_doc_ids = set()

## list year folders under base directory
def get_years(root: Path):
    if not root.exists():
        return []
    years = []
    for p in root.iterdir():
        if p.is_dir() and p.name.isdigit():
            years.append(p.name)
    return sorted(years)

YEARS = get_years(BASE_ORIG)
print("YEARS:", YEARS)

## diagnostic
doc_files = [p for p in BASE_ORIG.rglob("*.doc") if p.is_file()]

print("DOC files found:", len(doc_files))
doc_files

### Legacy .doc to .docx

In [ ]:
BASE_CONVERTED.mkdir(parents=True, exist_ok=True)

## conversion function
def convert_doc_to_docx(src_doc: Path, dst_docx: Path) -> tuple[bool, str]:
    try:
        dst_docx.parent.mkdir(parents=True, exist_ok=True)

        if dst_docx.exists() and dst_docx.stat().st_mtime >= src_doc.stat().st_mtime:
            return True, "skipped_up_to_date"

        ## build LibreOffice command
        cmd = [
            "soffice", "--headless", "--nologo", "--nofirststartwizard",
            "--convert-to", "docx",
            "--outdir", str(dst_docx.parent),
            str(src_doc),
        ]
        r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        expected = dst_docx.parent / (src_doc.stem + ".docx")

        if r.returncode != 0:
            return False, f"convert_error: {r.stderr.strip()[:300]}"
        if not expected.exists():
            return False, "convert_error: output_missing"

        if expected.resolve() != dst_docx.resolve():
            expected.replace(dst_docx)

        return True, "converted"
    except Exception as e:
        return False, f"convert_error: {str(e)}"


## initialize counters
converted = skipped = failed = 0
fail_samples = []

## processing loop
for y in YEARS:
    year_root = BASE_ORIG / y
    if not year_root.exists():
        continue

    ## find cv folders
    cv_folders = [
        p for p in year_root.rglob("*")
        if p.is_dir() and p.name.strip().lower() in CV_FOLDER_NAMES
    ]

    for folder in cv_folders:

        ## find .doc files
        for src_doc in folder.rglob("*.doc"):
            if not src_doc.is_file():
                continue

            ## destination path
            rel = src_doc.relative_to(BASE_ORIG)
            dst_docx = (BASE_CONVERTED / rel).with_suffix(".docx")

            ## conversion
            ok, msg = convert_doc_to_docx(src_doc, dst_docx)
            if ok and msg == "converted":
                converted += 1
            elif ok and msg == "skipped_up_to_date":
                skipped += 1
            else:
                failed += 1
                if len(fail_samples) < 5:
                    fail_samples.append((str(src_doc), msg))

## print summary
print(f"DOC→DOCX: converted={converted}, skipped_up_to_date={skipped}, failed={failed}")
if fail_samples:
    print("Sample failures:")
    for p, m in fail_samples:
        print(" ", p, "=>", m)

### Extraction

In [ ]:
## extract text from pdf using PyMuPDF (fitz)
def extract_text_pdf(path: Path, block_gap_threshold=14):
    try:
        output_pages = []

        with fitz.open(path) as doc:
            for page in doc:
                text_dict = page.get_text("dict")
                extracted_lines = []

                ## collect text lines from text blocks only
                for block in text_dict["blocks"]:
                    if block["type"] != 0:
                        continue

                    for line in block["lines"]:
                        spans = line["spans"]
                        line_text = "".join(span["text"] for span in spans).strip()

                        if not line_text:
                            continue

                        x0 = min(span["bbox"][0] for span in spans)
                        y0 = min(span["bbox"][1] for span in spans)
                        x1 = max(span["bbox"][2] for span in spans)
                        y1 = max(span["bbox"][3] for span in spans)

                        extracted_lines.append({
                            "text": line_text,
                            "x0": x0,
                            "y0": y0,
                            "x1": x1,
                            "y1": y1,
                        })

                ## sort lines in reading order
                extracted_lines.sort(key=lambda ln: (round(ln["y0"], 1), ln["x0"]))

                if not extracted_lines:
                    output_pages.append("")
                    continue

                page_parts = [extracted_lines[0]["text"]]
                prev_line = extracted_lines[0]

                for line in extracted_lines[1:]:
                    vertical_gap = line["y0"] - prev_line["y1"]

                    ## larger vertical gap = likely new block/section
                    if vertical_gap > block_gap_threshold:
                        page_parts.append("\n\n")
                    else:
                        page_parts.append("\n")

                    page_parts.append(line["text"])
                    prev_line = line

                output_pages.append("".join(page_parts).strip())

        return True, "\n\n".join(output_pages).strip(), None

    except Exception as e:
        return False, None, f"pdf_error: {str(e)}"

## extract text from docx using python-docx
def extract_text_docx(path: Path):
    try:
        doc = Document(path)
        text = "\n".join(p.text for p in doc.paragraphs)
        return True, text, None
    except Exception as e:
        return False, None, f"docx_error: {str(e)}"

## route file based on file extension
def extract_text_any(path: Path):
    ext = path.suffix.lower()
    if ext == ".pdf":
        return extract_text_pdf(path)
    if ext == ".docx":
        return extract_text_docx(path)

    ## failure if file type is not .pdf or .docx
    return False, None, f"unsupported_ext: {ext}"

## normalize for single-line .txt format
def normalize_text_for_txt(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\r\n", " ").replace("\r", " ").replace("\n", " ")
    text = " ".join(text.split())
    return text

## initialize rows, seen_doc_ids
rows = []
seen_doc_ids = set()

## scan (directory structure) and extract (text)
def scan_and_extract(scan_root: Path, canonical_root: Path, years: list[str]):
    """
    scan_root: where we look for files (BASE_CONVERTED or BASE_ORIG)
    canonical_root: used only to define doc_id consistently
    """

    ## processing loop
    for y in years:
        year_root = scan_root / y
        if not year_root.exists():
            continue

        ## find cv folders
        cv_folders = [
            p for p in year_root.rglob("*")
            if p.is_dir() and p.name.strip().lower() in CV_FOLDER_NAMES
        ]
        if not cv_folders:
            continue

        ## iterate through cv folders
        for folder in cv_folders:
            for f in folder.rglob("*"):
                if not f.is_file():
                    continue
                if f.suffix.lower() not in ALLOWED_EXTS:
                    continue

                ## relative path as unique identifier
                canonical_rel = f.relative_to(canonical_root)
                doc_id = str(canonical_rel)

                ## skip if file already processed
                if doc_id in seen_doc_ids:
                    continue
                seen_doc_ids.add(doc_id)

                ## extract
                ok, text, reason = extract_text_any(f)
                if ok and text:
                    text = str(text).replace("\r\n", "\n").replace("\r", "\n").strip()

                ## metadata for corpus
                rows.append({
                    "year": y,
                    "doc_id": doc_id,
                    "path": str(f),
                    "ext": f.suffix.lower(),
                    "ok": ok,
                    "fail_reason": reason,
                    "n_chars": len(text) if text else 0,
                    "n_words": len(re.findall(r"\S+", text)) if text else 0,
                    "text": text if ok else None
                })

## converted base first (preferred)
scan_and_extract(BASE_CONVERTED, BASE_CONVERTED, YEARS)

## original data second (pdf + native docx)
scan_and_extract(BASE_ORIG, BASE_ORIG, YEARS)

## Corpus table

In [ ]:
## build corpus table
df_corpus = pd.DataFrame(rows)

### Corpus table quality checks

In [ ]:
## examine aggregation quality
print("total rows:", len(df_corpus))
print("success vs fail:\n", df_corpus["ok"].value_counts())

## examine dataframe
print("rows:", len(df_corpus))
print("\nok vs fail:\n", df_corpus["ok"].value_counts(dropna=False))
print("\nfailure reasons:\n", df_corpus.loc[df_corpus["ok"] == False, "fail_reason"].value_counts().head(10))

## preview
cols = ["year", "doc_id", "ext", "ok", "n_chars", "n_words", "fail_reason", "text"]
df_corpus.reindex(columns=cols).head(20)

In [ ]:
## copy of successfully extracted documents
df_ok = df_corpus[df_corpus["ok"] == True].copy()

## random doc for quality check
row = df_ok[df_ok["ext"] == ".docx"].sample(1, random_state=7).iloc[0]

## diagnostic of random doc
print("DOC ID:", row["doc_id"])
print("PATH:", row["path"])
print("CHARS:", row["n_chars"], "| WORDS:", row["n_words"])

In [ ]:
## get extracted text from document row
text = row["text"]

## display text in scrollable html container for evaluation
display(HTML(f"""
<div style="
    height:600px;
    overflow:auto;
    border:1px solid #ccc;
    padding:10px;
    white-space:pre-wrap;
    font-family:monospace;
">
{text}
</div>
"""))

### Corpus table visualizations

In [ ]:
## box plot
## remove rows where n_words or year values are missing
plot_df = df_corpus.dropna(subset=["n_words", "year"])

## year column to string, sort
plot_df["year"] = plot_df["year"].astype(str)
years = sorted(plot_df["year"].unique())

## create list of word count arrays grouped by year
data = [plot_df.loc[plot_df["year"] == y, "n_words"] for y in years]

## plot
plt.figure(figsize=(10, 5))
plt.boxplot(data, labels=years)
plt.title("Distribution of Word Count by Year")
plt.xlabel("Year")
plt.ylabel("Word Count")
plt.tight_layout()
plt.show()

In [ ]:
## violin plot
## remove rows where n_words or year values are missing
plot_df = df_corpus.dropna(subset=["n_words", "year"])

## year column to string, sort
plot_df["year"] = plot_df["year"].astype(str)
years = sorted(plot_df["year"].unique())

## create list of word count arrays grouped by year
data = [plot_df.loc[plot_df["year"] == y, "n_words"] for y in years]

## plot
plt.figure(figsize=(10, 5))
plt.violinplot(data, showmedians=True)
plt.xticks(range(1, len(years) + 1), years)
plt.title("Distribution of Word Count by Year")
plt.xlabel("Year")
plt.ylabel("Word Count")
plt.tight_layout()
plt.show()

## Text from corpus table to actual `.txt`

In [ ]:
## output root directory
OUT_TXT_ROOT = Path("/content/drive/MyDrive/youthnex_txt")
OUT_TXT_ROOT.mkdir(parents=True, exist_ok=True)

## compute sha256 has of text if it does not exist
def _sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8", errors="ignore")).hexdigest()

## write file using temp file then replace
def _atomic_write(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(content, encoding="utf-8", errors="ignore")
    tmp.replace(path)

## initialize counters
written, skipped = 0, 0

## keep successful extractions only
df_ok = df_corpus[df_corpus["ok"] == True].copy()

## iterate through documents
for _, row in df_ok.iterrows():
    doc_id = row["doc_id"]
    text = row["text"] or ""

    ## mirror folder structure but swap to .txt
    out_txt = OUT_TXT_ROOT / Path(doc_id).with_suffix(".txt")
    out_meta = out_txt.with_suffix(".json")

    ## preserve line structure
    text = str(text).replace("\r\n","\n").replace("\r","\n").strip()

    ## text hash
    text_hash = _sha256_text(text)

    ## skip if file already exists and hash matches
    if out_txt.exists() and out_meta.exists():
        try:
            meta = json.loads(out_meta.read_text(encoding="utf-8"))
            if meta.get("text_sha256") == text_hash:
                skipped += 1
                continue
        except Exception:
            pass

    ## write text file
    _atomic_write(out_txt, text)

    ## metadata dictionary
    meta = {
        "doc_id": doc_id,
        "source_path": row["path"],
        "year": row["year"],
        "ext": row["ext"],
        "extracted_at_epoch": int(time.time()),
        "text_sha256": text_hash,
        "n_chars": int(row.get("n_chars", len(text))),
        "n_words": int(row.get("n_words", len(text.split()))),
    }

    ## metadata file
    _atomic_write(out_meta, json.dumps(meta, indent=2))

    ## increment counter
    written += 1

## summary
print(f"Wrote {written} .txt files | skipped {skipped} (already current) | total ok {len(df_ok)}")
print("Output root:", OUT_TXT_ROOT)